In [ ]:
import __main__
import sys, os
import time
project_root = os.path.abspath("..")  # adjust if notebook is elsewhere
sys.path.insert(0, project_root)
from typing import Dict, List, Literal, Tuple, Optional, Any, Union
import logging
import random
from dataclasses import dataclass

import category_encoders as ce
import matplotlib.pyplot as plt

import numexpr as ne # makes numpy operations faster
import numpy as np
import pandas as pd
import polars as pl
from tqdm import tqdm

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score, mean_absolute_error, root_mean_squared_error, r2_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neighbors import NearestNeighbors, KernelDensity
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.random_projection import GaussianRandomProjection

from catboost import CatBoostRegressor, CatBoostClassifier

from geo_functions import compute_cyclicity_score, split_dataset_to_linear_and_cyclic, scale_train_and_test_sets, Windowing

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    # print(torch.cuda.memory_reserved(0) / 1e6, "MB reserved")
    # print(torch.cuda.memory_allocated(0) / 1e6, "MB allocated")

import src.param_config.config_paths as P

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.info("Starting process...")
logging.warning("Something looks off...")
logging.error("Something failed.")


In [ ]:
"California housing dataset (tabular) -  has latitude/longitude info"

from sklearn.datasets import fetch_california_housing

# as_frame=True returns a Pandas DataFrame immediately
data = fetch_california_housing(as_frame=True)
df = data.frame

X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"].values[:, None] # made to 2D array


In [ ]:
"Bike sharing dataset (https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)"
from ucimlrepo import fetch_ucirepo 
  
bike_sharing = fetch_ucirepo(id=275) 
  
X = bike_sharing.data.features 
y = bike_sharing.data.targets
# print(bike_sharing.variables) 

X = X.drop(["dteday"], axis=1)

cyclic_cols = ['season', 'mnth', 'hr', 'weekday']
linear_cols = ['yr', 'holiday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']

X_cyclic = X[cyclic_cols]
X_linear = X[linear_cols]


In [ ]:
"[works] china weather"

file_location = '../public_datasets/3D/china_weather/china_weather_000.npy'
weather_array = np.load(file_location).transpose(0, 2, 1)

for col in range(weather_array.shape[2]):
    data_col    = weather_array[2, :, col].flatten()
    cycle_score = compute_cyclicity_score(data_col)
    print(f"col {col} cycl. score: {cycle_score:.4f}")

y_indices       = [7, 10] #[4, 5, 6, 7, 10]

mask            = np.ones(weather_array.shape[2], dtype=bool)
mask[y_indices] = False
X_cut           = weather_array[:, :, mask]      # (stations, timesteps, n_features)
y_cut           = weather_array[:, :, y_indices] # (stations, timesteps, n_targets)

assert X_cut.shape[2] + y_cut.shape[2] == weather_array.shape[2]

page_choice = 7
X = pd.DataFrame(X_cut[page_choice])
y = y_cut[page_choice]

print(f"X shape: {X.shape}, y shape: {y.shape}")


In [ ]:
"[works] longterm weather (https://www.kaggle.com/datasets/alistairking/weather-long-term-time-series-forecasting)"

df = pd.read_csv('../public_datasets/2D/tabular/longterm_weather/longterm_weather.csv')

for col in df.columns:
    if df[col].dtype not in [np.float64, np.float32, np.int64, np.int32]:
        continue
    cycle_score = compute_cyclicity_score(df[col].to_numpy(), )
    print(f"col {col} cycl. score: {cycle_score:.4f}")

y_cols = ["rain"]
y      = df[y_cols].to_numpy()
X      = df.drop(columns=y_cols, inplace=False)


In [ ]:
"another weather (https://www.kaggle.com/datasets/muthuj7/weather-dataset?select=weatherHistory.csv)"

df = pd.read_csv('../public_datasets/2D/tabular/another_weather/weatherHistory.csv')

for col in df.columns:
    if df[col].dtype not in [np.float64, np.float32, np.int64, np.int32]:
        continue
    cycle_score = compute_cyclicity_score(df[col].to_numpy())
    print(f"Cyclicity score of feature {col}: {cycle_score}")


In [ ]:
"cities weather (https://www.kaggle.com/datasets/selfishgene/historical-hourly-weather-data?resource=download)"

weather_folder   = "../public_datasets/2D/tabular/Hourly Weather Data 2012-2017"
files_to_combine = ["temperature.csv", "humidity.csv", "wind_speed.csv", "pressure.csv", "wind_direction.csv"]

dfs = {f.split(".")[0]: pd.read_csv(os.path.join(weather_folder, f)) for f in files_to_combine}

# Extract city names (assumes all files have same columns)
cities     = [col for col in dfs["temperature"].columns if col != "datetime"]
timesteps  = len(dfs["temperature"])
properties = len(files_to_combine)

weather_array = np.zeros((len(cities), timesteps, properties), dtype=float)

# Fill array
for p, prop in enumerate(files_to_combine):
    df_prop = dfs[prop.split(".")[0]]  # remove .csv from name
    for c, city in enumerate(cities):
        weather_array[c, :, p] = df_prop[city].values

for c in range(weather_array.shape[0]):        # cities
    for p in range(weather_array.shape[2]):    # properties
        col = weather_array[c, :, p]
        if np.isnan(col).any():
            mean_val = np.nanmean(col)  # compute mean ignoring NaNs
            col[np.isnan(col)] = mean_val
            weather_array[c, :, p] = col
print("Array shape:", weather_array.shape)
print("NaNs remaining:", np.isnan(weather_array).sum())

cycle_score = compute_cyclicity_score(weather_array[0, :, 0])
print(f"Cyclicity score : {cycle_score}")



In [ ]:
"Pseudo-cyclic synthetic dataset (https://archive.ics.uci.edu/dataset/136/pseudo+periodic+synthetic+time+series)"

data = np.loadtxt('../public_datasets/2D/tabular/pseudo_cyclic/synthetic.data')

for i in range(data.shape[1]):
    cycle_score = compute_cyclicity_score(data[:,i])
    print(f"Cyclicity score of feature {i}: {cycle_score}")

In [ ]:
"Traffic flow dataset (https://archive.ics.uci.edu/dataset/608/traffic+flow+forecasting)"

from scipy.io import loadmat

# Load data
data = loadmat("../public_datasets/2D/tabular/traffic_dataset/traffic_dataset.mat")

def flatten_X(mat_array):
    """
    Convert 1xN MATLAB object array of 36x48 matrices into 2D numeric array
    N rows, 36*48 columns
    """
    flattened = []
    for m in mat_array[0]:
        # Ensure numeric type
        flattened.append(np.array(m, dtype=float).reshape(-1))
    return np.stack(flattened, axis=0)

# Flatten input features
X_train_np = flatten_X(data['tra_X_tr'])
X_test_np  = flatten_X(data['tra_X_te'])

# Outputs: already numeric, just transpose to N x 36
y_train_np = data['tra_Y_tr'].T.astype(float)
y_test_np  = data['tra_Y_te'].T.astype(float)

# Convert to Polars
X_train = pl.DataFrame(X_train_np)
X_test  = pl.DataFrame(X_test_np)
y_train = pl.DataFrame(y_train_np)
y_test  = pl.DataFrame(y_test_np)

print(X_train.shape, y_train.shape)
print(X_train.head())
print(y_train.head())



In [ ]:
"Electric power data (https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption)"

from ucimlrepo import fetch_ucirepo

def get_household_power_consumption(target: str = "Global_active_power") -> Tuple[pl.DataFrame, pl.Series]:
    # 1. Fetch dataset
    ds = fetch_ucirepo(id=235)
    
    # 2. Combine and clean in Pandas to handle mixed types ('?')
    # For this dataset, ds.data.original contains all columns combined
    df_pd = ds.data.original.copy()
    
    # Coerce all columns to numeric except Date and Time
    for col in df_pd.columns:
        if col not in ["Date", "Time"]:
            df_pd[col] = pd.to_numeric(df_pd[col], errors='coerce')
            
    # 3. Convert to Polars safely
    df = pl.from_pandas(df_pd)
    
    # 4. Handle target and nulls
    if target not in df.columns:
        raise ValueError(f"Target '{target}' not found")
        
    df = df.filter(pl.col(target).is_not_null())
    y = df.get_column(target)
    X = df.drop(target).fill_null(0)
    return X, y

X, y = get_household_power_consumption()
print(f"X shape: {X.shape}, y shape: {y.shape}")


numeric_types = [pl.Float32, pl.Float64, pl.Int32, pl.Int64]

for col in X.columns:
    if X[col].dtype not in numeric_types:
        continue
    cycle_score = compute_cyclicity_score(X[col].to_numpy())
    print(f"Cyclicity score of feature {col}: {cycle_score}")



In [ ]:
"[RUN] utils functions"

# for tabular
class EuclidEncoder(nn.Module):
    def __init__(self, window_size, input_dim, z_dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(window_size * input_dim, hidden),
            nn.ReLU())
            # nn.SiLU())
        self.mu     = nn.Linear(hidden, z_dim)
        self.logvar = nn.Linear(hidden, z_dim)

    def forward(self, x):
        h = self.net(x)
        return self.mu(h), self.logvar(h)

class SphericalEncoder(nn.Module):
    def __init__(self, window_size, input_dim, z_dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(window_size * input_dim, hidden),
            nn.ReLU())
            # nn.SiLU())

        self.mu_raw = nn.Linear(hidden, z_dim)
        self.kappa  = nn.Linear(hidden, 1)

    def forward(self, x):
        h      = self.net(x)
        mu_dir = F.normalize(self.mu_raw(h), dim=-1)
        kappa  = F.softplus(self.kappa(h)) + 1e-3
        return mu_dir, kappa

class Decoder(nn.Module):
    def __init__(self, z_dim_total, window_size, output_dim, hidden):
        super().__init__()
        self.window_size = window_size
        self.output_dim   = output_dim  # number of features (linear+cyclic)
        self.net = nn.Sequential(
            nn.Linear(z_dim_total, hidden),
            nn.ReLU(),
            # nn.SiLU(),
            nn.Linear(hidden, window_size * output_dim))  # << flattened output

    def forward(self, z):
        # z: (B, z_dim_total)
        return self.net(z)  # shape (B, window_size*output_dim)


# for timeseries
class LSTMEncoderEuclid(nn.Module):
    """Euclidean latent LSTM encoder (Gaussian z_e)."""
    def __init__(self, input_dim, hidden_dim, z_dim):
        super().__init__()
        self.lstm   = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.mu     = nn.Linear(hidden_dim, z_dim)
        self.logvar = nn.Linear(hidden_dim, z_dim)

    def forward(self, x):
        # x: [B, T, D]
        _, (lstm_hidden, _) = self.lstm(x)           # h_n: [1, B, H]
        encoder_hidden      = lstm_hidden.squeeze(0) # [B, H]
        return self.mu(encoder_hidden), self.logvar(encoder_hidden)  # [B, z_dim], [B, z_dim]

class LSTMSphericalEncoder(nn.Module):
    """Spherical latent LSTM encoder (vMF z_s)."""
    def __init__(self, input_dim, hidden_dim, z_dim):
        super().__init__()
        self.lstm   = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.mu_raw = nn.Linear(hidden_dim, z_dim)
        self.kappa  = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        h      = h_n.squeeze(0)
        mu_dir = F.normalize(self.mu_raw(h), dim=-1)
        kappa  = F.softplus(self.kappa(h)) + 1e-3
        return mu_dir, kappa

class MLPDecoder(nn.Module):
    """Decode concatenated latent vector z_e + z_s -> windowed features."""
    def __init__(self, z_dim_total, window_size, output_dim, hidden_dim):
        super().__init__()
        self.window_size = window_size
        self.output_dim  = output_dim
        self.net = nn.Sequential(
            nn.Linear(z_dim_total, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, window_size * output_dim)  # flattened output
        )

    def forward(self, z):
        # z: [B, z_dim_total]
        return self.net(z)  # [B, window_size*output_dim]


class Reparam:
    @staticmethod
    def reparam_gaussian(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        """Gaussian reparameterization."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    @staticmethod #old, maybe remove
    def reparam_vmf(mu_dir: torch.Tensor, kappa: torch.Tensor) -> torch.Tensor:
        """vMF has no closed form, so approximate vMF sampling: Gaussian noise + projection. Radius = 1 by construction.
        This is what Davidson (2018) does as well, fine because ELBO needs approximate sampling"""
        eps = torch.randn_like(mu_dir)
        z   = mu_dir + eps / (kappa + 1e-6)
        return F.normalize(z, dim= -1) #this projects to unit sphere

    @staticmethod
    def sample_vmf(mu: torch.Tensor, kappa: torch.Tensor) -> torch.Tensor:
        """Sample from vMF distribution on S^{d-1} with direction mu and concentration kappa.
        mu: (batch, dim), must be unit norm
        kappa: (batch, 1)
        returns: (batch, dim)"""
        batch, dim = mu.shape
        # for very small kappa, approximate uniform sampling
        eps = torch.rand(batch, 1, device=mu.device)
        w   = 1 + (torch.log(eps + (1 - eps) * torch.exp(-2*kappa))) / kappa  # simplified
        w   = w.clamp(-1+1e-7, 1-1e-7)
        
        # sample v ~ uniform on unit sphere orthogonal to mu
        v = torch.randn(batch, dim, device=mu.device)
        v = v - (v*mu).sum(dim=1, keepdim=True) * mu  # make orthogonal
        v = F.normalize(v, dim=1)
        
        # combine
        z = w * mu + torch.sqrt(1 - w**2) * v
        return F.normalize(z, dim=-1)


def kl_gaussian(mu, logvar):
    """KL divergence between Gaussian distr. and standard normal distr."""
    return -0.5 * torch.sum(1 + logvar - mu**2 - logvar.exp(), dim=1).mean()

# remove if kl_vmf_uniform works
def regularization_vmf(kappa, dim):
    """Spherical has no closed form of the KL term, so this functino is a proxy regularizer
    This is more of a concentration regularizer encouraging proximity to a uniform hyperspherical prior than a KL divergence."""
    # return (kappa - (dim - 1) * torch.log(kappa + 1e-6)).mean() # my old one
    return (kappa**2).mean()

def kl_vmf_uniform(mu: torch.Tensor, kappa: torch.Tensor) -> torch.Tensor:
    """Approximate KL(vMF(mu, kappa) || uniform(S^{d-1})), from Davidson et al. 2018
    mu: (batch, dim)
    kappa: (batch, 1)"""
    d     = mu.shape[1]
    kappa = kappa.clamp_min(1e-3) # avoid log(0)
    if not torch.is_tensor(kappa):
        kappa = torch.tensor(kappa, device=mu.device)
    # Approximate log C_d(kappa)
    log_c = (d/2 - 1) * torch.log(kappa) - (d/2) * torch.log(torch.tensor(2*torch.pi, device=mu.device)) - kappa
    kl    = kappa.squeeze(-1) - log_c  # per-batch, KL ≈ kappa * (μ · μ) + log C_d(kappa)  (simplified)
    return kl.mean()


class MixedEncoder(nn.Module):
    """Encoder producing mixed latent variables:
      - Euclidean (Gaussian) z_e with mean `mu_e` and log-variance `logvar_e`
      - Hyperspherical (vMF) z_s with direction `mu_s` and concentration `kappa`
    Args:
        input_dim (int): Number of input features
        hidden (int): Size of shared hidden layers
        z_e_dim (int): Dimension of Euclidean latent z_e
        z_s_dim (int): Dimension of hyperspherical latent z_s"""
    def __init__(self, input_dim, hidden_dim, z_e_dim, z_s_dim):
        super().__init__()
        # Shared hidden layers
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU())
        # Euclidean latent parameters
        self.mu_e    = nn.Linear(hidden_dim, z_e_dim) # mean vector (euclidean)
        self.logvar_e= nn.Linear(hidden_dim, z_e_dim) # log variance (euclidean)
        # Hyperspherical latent parameters
        self.mu_s    = nn.Linear(hidden_dim, z_s_dim) # mean direction (spherical)
        self.kappa   = nn.Linear(hidden_dim, 1)       # vMF concentration param (spherical)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """Forward pass.
        Args: x (Tensor): Input tensor of shape [batch_size, input_dim]
        * hidden_layer (Tensor) is a shared representation computed as self.shared(x)
        Returns:
            mu_e (Tensor): Mean of Gaussian latent z_e [batch_size, z_e_dim]
            logvar_e (Tensor): Log-variance of z_e [batch_size, z_e_dim]
            mu_s (Tensor): Unit vector direction of vMF latent z_s [batch_size, z_s_dim]
            kappa (Tensor): Concentration of vMF latent z_s [batch_size, 1], positive"""
        hidden_layer = self.shared(x)              # Shared hidden representation
        mu_e         = self.mu_e(hidden_layer)     # Gaussian mean
        logvar_e     = self.logvar_e(hidden_layer) # Gaussian log-variance
        mu_s         = F.normalize(self.mu_s(hidden_layer), dim=-1) # Unit vector for vMF
        kappa        = F.softplus(self.kappa(hidden_layer)) + 1e-3  # Positive concentration
        return mu_e, logvar_e, mu_s, kappa


class WithSplit:
    @staticmethod
    @torch.no_grad()
    def encode_tabular_dataset(lin_windows, cyc_windows, lin_encoder, cyc_encoder, pooling="mean"):
        """Encode linear + cyclic windows.
        lin_windows: (num_windows, win, D_lin)
        cyc_windows: (num_windows, win, D_cyc)"""
        N, win, D_lin  = lin_windows.shape
        D_cyc          = cyc_windows.shape[-1]

        lin_flat       = lin_windows.view(N, win*D_lin)
        cyc_flat       = cyc_windows.view(N, win*D_cyc)

        mu_e, logvar_e = lin_encoder(lin_flat)
        mu_s, kappa    = cyc_encoder(cyc_flat)

        z = torch.cat([mu_e, mu_s], dim=-1)

        if pooling is None:
            return z.view(N, 1, -1)  # add dummy W dim
        if pooling == "mean":
            return z.mean(dim=0, keepdim=True)  # only one sequence
        if pooling == "max":
            return z.max(dim=0, keepdim=True).values
        raise ValueError(f"Unknown pooling: {pooling}")

    @staticmethod
    @torch.no_grad()
    def encode_timeseries_dataset(X_lin_win, X_cyc_win, lin_encoder, cyc_encoder, pooling="mean"):
        """Encode linear + cyclic windows.
        X_lin_win: (num_windows, win, D_lin)
        X_cyc_win: (num_windows, win, D_cyc)"""
        num_windows, win, D_lin  = X_lin_win.shape
        D_cyc          = X_cyc_win.shape[-1]
        mu_e, logvar_e = lin_encoder(X_lin_win)
        mu_s, kappa    = cyc_encoder(X_cyc_win)

        z = torch.cat([mu_e, mu_s], dim=-1)

        if pooling is None:
            return z.view(num_windows, 1, -1)  # add dummy W dim
        if pooling == "mean":
            return z.mean(dim=0, keepdim=True)  # only one sequence
        if pooling == "max":
            return z.max(dim=0, keepdim=True).values
        raise ValueError(f"Unknown pooling: {pooling}")

    @staticmethod
    def vae_train_step_for_tabular(x_lin: torch.Tensor, x_cyc: torch.Tensor, lin_encoder: nn.Module, cyc_encoder: nn.Module,
                    decoder: nn.Module, lambdas: dict) -> torch.Tensor:
        """Single VAE training step.
        - x_lin, x_cyc : (batch, window_size, features)"""
        B, win, D_lin = x_lin.shape
        D_cyc         = x_cyc.shape[-1]

        x_lin_flat    = x_lin.view(B, win*D_lin)
        x_cyc_flat    = x_cyc.view(B, win*D_cyc)

        mu_e, logvar_e= lin_encoder(x_lin_flat)
        mu_s, kappa   = cyc_encoder(x_cyc_flat)
        
        # ensure kappa is Tensor
        if not isinstance(kappa, torch.Tensor):
            kappa = torch.tensor(kappa, dtype=torch.float32, device=x_cyc.device)

        z_e = Reparam.reparam_gaussian(mu_e, logvar_e)
        z_s = Reparam.reparam_vmf(mu_s, kappa)
        z   = torch.cat([z_e, z_s], dim=-1)

        x_hat       = decoder(z)
        x_full_flat = torch.cat([x_lin_flat, x_cyc_flat], dim=-1)
        L_recon     = F.mse_loss(x_hat, x_full_flat)

        L_kl_e   = kl_gaussian(mu_e, logvar_e)
        # L_reg_s  = regularization_vmf(kappa, z_s.size(-1))
        L_kl_s   = kl_vmf_uniform(mu_s, kappa)  # replaces old regularization_vmf()

        # return lambdas["reconstr"]*L_recon + lambdas["euc"]*L_kl_e + lambdas["sph"]*L_reg_s
        return lambdas["reconstr"]*L_recon + lambdas["euc"]*L_kl_e + lambdas["sph"]*L_kl_s

    @staticmethod
    def vae_train_step_for_timeseries(x_lin: torch.Tensor, x_cyc: torch.Tensor,
                    lin_encoder: nn.Module, cyc_encoder: nn.Module,
                    decoder: nn.Module, lambdas: dict) -> torch.Tensor:
        """Single VAE training step for split latent VAE.
        - x_lin, x_cyc : (batch, window_size, features)
        - decoder expects concatenated z_e + z_s of shape (batch, z_total)
        and outputs flattened reconstruction of shape (batch, window_size*D_total)"""
        B, win, D_lin = x_lin.shape
        D_cyc         = x_cyc.shape[-1]

        # flatten per-window inputs for LSTM
        x_lin_flat = x_lin
        x_cyc_flat = x_cyc

        # ---- encode ----
        mu_e, logvar_e = lin_encoder(x_lin_flat)
        mu_s, kappa    = cyc_encoder(x_cyc_flat)

        # ---- reparameterize ----
        z_e = Reparam.reparam_gaussian(mu_e, logvar_e)  # [B, z_e]
        z_s = Reparam.reparam_vmf(mu_s, kappa)         # [B, z_s]
        z   = torch.cat([z_e, z_s], dim=-1)  # [B, z_total]

        # ---- decode ----
        x_hat = decoder(z)                  # [B, window_size * D_total]

        # ---- flatten original inputs for reconstruction loss ----
        x_full_flat = torch.cat([x_lin_flat.reshape(B, -1), x_cyc_flat.reshape(B, -1)], dim=-1)

        # ---- losses ----
        L_recon = F.mse_loss(x_hat, x_full_flat)
        L_kl_e  = kl_gaussian(mu_e, logvar_e)
        L_kl_s  = kl_vmf_uniform(mu_s, kappa)
        loss    = lambdas["reconstr"] * L_recon + lambdas["euc"] * L_kl_e + lambdas["sph"] * L_kl_s
        return loss

    @staticmethod
    def train_linear_and_cyclic_vaes_for_1_epoch_timeseries(data_loader: DataLoader, lin_encoder: nn.Module, cyc_encoder: nn.Module,
                    decoder: nn.Module, optimizer: torch.optim.Optimizer, lambdas: dict):
        """Full training loop over one epoch for the VAE.
        Args:
            data_loader : DataLoader yielding (x_lin, x_cyc). note x_lin and x_cyc are separated
            lin_encoder : Linear encoder
            cyc_encoder : Cyclic encoder
            decoder     : Decoder
            optimizer   : Optimizer
            lambdas     : dict of loss weights"""
        lin_encoder.train()
        cyc_encoder.train()
        decoder.train()

        epoch_loss = 0
        for x_linear, x_cyclic in data_loader:
            optimizer.zero_grad()
            loss = WithSplit.vae_train_step_for_timeseries(x_linear, x_cyclic, lin_encoder, cyc_encoder, decoder, lambdas)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        epoch_loss /= len(data_loader)
        return epoch_loss

    @staticmethod
    def train_linear_and_cyclic_vaes_for_1_epoch_tabular(data_loader: DataLoader, lin_encoder: nn.Module, cyc_encoder: nn.Module,
                    decoder: nn.Module, optimizer: torch.optim.Optimizer, lambdas: dict):
        """Full training loop over one epoch for the VAE.
        Args:
            data_loader : DataLoader yielding (x_lin, x_cyc). note x_lin and x_cyc are separated
            lin_encoder : Linear encoder
            cyc_encoder : Cyclic encoder
            decoder     : Decoder
            optimizer   : Optimizer
            lambdas     : dict of loss weights"""
        lin_encoder.train()
        cyc_encoder.train()
        decoder.train()

        epoch_loss = 0
        for x_linear, x_cyclic in data_loader:
            optimizer.zero_grad()
            loss = WithSplit.vae_train_step_for_tabular(x_linear, x_cyclic, lin_encoder, cyc_encoder, decoder, lambdas)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        epoch_loss /= len(data_loader)
        return epoch_loss


class NoSplit:
    @staticmethod
    @torch.no_grad()
    def encode_dataset_no_split(X_win: torch.Tensor, encoder: nn.Module, pooling: str | None = "mean") -> torch.Tensor:
        """Encode a dataset without linear/cyclic split.
        x : (num_windows, window_size, num_features)
        pooling : "mean" or None
        Returns : (num_windows, z_dim) if pooled, else (num_windows, window_size, z_dim)"""
        N_w, win, D = X_win.shape
        x_flat      = X_win.view(N_w, win*D)
        mu, _       = encoder(x_flat)
        
        if pooling is None:
            return mu
        if pooling == "mean":
            return mu.mean(dim=0, keepdim=True)  # average over windows
        raise ValueError(f"Unknown pooling: {pooling}")

    @staticmethod
    def vae_train_step_no_split(x: torch.Tensor, encoder: nn.Module, decoder: nn.Module, lambdas: dict) -> torch.Tensor:
        """Single VAE step for dataset without linear/cyclic split.
        x : (num_windows, window_size, num_features)"""
        N_w, win, D = x.shape
        x_flat      = x.view(N_w, win*D)

        mu, logvar  = encoder(x_flat)
        z           = Reparam.reparam_gaussian(mu, logvar)

        x_hat   = decoder(z)
        L_recon = F.mse_loss(x_hat, x_flat)
        L_kl    = kl_gaussian(mu, logvar)

        return lambdas["reconstr"]*L_recon + lambdas["euc"]*L_kl

    @staticmethod
    def train_vae_no_split(loader, encoder, decoder, optimizer, lambdas):
        encoder.train(); decoder.train()
        epoch_loss = 0
        for x, in loader:
            optimizer.zero_grad()
            loss = NoSplit.vae_train_step_no_split(x, encoder, decoder, lambdas)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        epoch_loss /= len(loader)
        return epoch_loss


# general
def early_stop(current_loss: float, best_loss: float, counter: int, patience: int = 5):
    """Quick early stopping tracker.
    Returns updated best_loss, counter, and stop flag."""
    if current_loss < best_loss:
        return current_loss, 0, False
    else:
        counter += 1
        stop = counter >= patience
        return best_loss, counter, stop


def fit_catboost_multi(X_train: Union[np.ndarray, torch.Tensor], y_train: Union[np.ndarray, torch.Tensor],
                       X_test: Union[np.ndarray, torch.Tensor],) -> np.ndarray:
    """Fit CatBoost for multi-output regression."""
    if isinstance(X_train, torch.Tensor):
        X_train = X_train.detach().cpu().numpy()
    if isinstance(y_train, torch.Tensor):
        y_train = y_train.detach().cpu().numpy()
    if isinstance(X_test, torch.Tensor):
        X_test = X_test.detach().cpu().numpy()

    model = MultiOutputRegressor(CatBoostRegressor(verbose=0))
    model.fit(X_train, y_train)
    return model.predict(X_test)


def estimate_entropy(x, bandwidth: float = 0.2) -> float:
    """KDE-based differential entropy estimator for 1D samples."""
    x = np.asarray(x)
    if x.ndim != 1:
        raise ValueError("estimate_entropy expects 1D input")
    x   = x[:, None]
    kde = KernelDensity(bandwidth=bandwidth).fit(x)
    return -kde.score_samples(x).mean()

def pool_latents(z: torch.Tensor, z_e_dim: int) -> torch.Tensor:
    """Pool mixed latents over windows, depending on latent type (spherical vs Euclidean)
    z: (batch, windows, z_total)
    z_e_dim: dimension of Euclidean part
    NOTE: assumes z is concat like so: [linear, cyclic]"""
    z_euclid = z[..., :z_e_dim]  # Euclidean
    z_spher  = z[..., z_e_dim:]  # Spherical
    z_e_mean = z_euclid.mean(dim=1)  # Euclidean mean
    z_s_mean = F.normalize(z_spher.sum(dim=1), dim=-1)  # Spherical mean
    return torch.cat([z_e_mean, z_s_mean], dim=-1)


In [ ]:
"[RUN] Params"
dataset     = "bike_sharing" # longterm_weather electric_power china_weather cali_housing bike_sharing

prediction_task = "tabular"  # forecast nowcast tabular
window_size = 64
sliding_size= 8
if prediction_task == "tabular":
    window_size  = 1
    sliding_size = 1

epochs      = 50
z_dim_total = 32 #16
hidden_dim  = 60
lr_optimizer= 1e-3
batch_size  = 256
earlystop_patience = 8

# for split scenario
hidden_dim_split= int(hidden_dim / np.sqrt(2)) # approximation of (§2.1 of https://arxiv.org/pdf/2001.08361)
z_dim_euclid    = z_dim_total // 2
z_dim_spheric   = z_dim_total - z_dim_euclid

"Shared preprocessing steps"
cols_to_drop = {"longterm_weather": ["date"],
                "electric_power":   ["Date", "Time"],
                "china_weather":    [],
                "cali_housing":     [],
                "bike_sharing":     ["dteday"]}
cyclic_threshold_dict = {"longterm_weather": 0.52,
                         "electric_power":   0.5,
                         "china_weather":    0.25,
                         "cali_housing":     0.3,
                         "bike_sharing":     0.3}

X.drop(columns=cols_to_drop[dataset], inplace=True, errors='ignore') # drop non-numeric if present
cyclic_threshold = cyclic_threshold_dict[dataset]

# 1. train/test split without shuffling (time series)
if prediction_task == "tabular": # shuffle tabular
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)
else: # dont shuffle timeseries
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False) #timeseries order matters


In [ ]:
"DIRECT PREDICTION"

def make_direct_prediction(X_train: np.ndarray, X_test: np.ndarray, y_train: np.ndarray, y_test: np.ndarray,
                           window_size: int, sliding_size: int, task: str,) -> tuple[float, float]:
    """Direct CatBoost baseline with task-consistent windowing."""

    print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

    if task == "tabular":
        X_train_flat, X_test_flat = X_train, X_test
        y_train_w, y_test_w       = y_train, y_test
    else:
        # --- window X ---
        y_train_w = Windowing.make_windows_from_y(y_train, window_size, sliding_size, task=task)
        y_test_w  = Windowing.make_windows_from_y(y_test, window_size, sliding_size, task=task)

        # Use torch.from_numpy since input is already np.ndarray
        X_train_w = Windowing.make_windows_from_X(torch.from_numpy(X_train).float(), window_size, sliding_size)
        X_test_w  = Windowing.make_windows_from_X(torch.from_numpy(X_test).float(), window_size, sliding_size)

        X_train_flat = X_train_w.reshape(X_train_w.shape[0], -1).numpy()
        X_test_flat  = X_test_w.reshape(X_test_w.shape[0], -1).numpy()

    # --- scale ---
    X_train_flat, X_test_flat = scale_train_and_test_sets(X_train_flat, X_test_flat)

    y_train_prep = y_train_w.reshape(-1, 1) if y_train_w.ndim == 1 else y_train_w
    y_test_prep  = y_test_w.reshape(-1, 1) if y_test_w.ndim == 1 else y_test_w
    y_train_scaled, y_test_scaled = scale_train_and_test_sets(y_train_prep, y_test_prep)

    # --- regression ---
    y_hat = fit_catboost_multi(X_train_flat, y_train_scaled, X_test_flat)
    rmse  = root_mean_squared_error(y_test_scaled, y_hat)
    r2    = r2_score(y_test_scaled, y_hat)
    return rmse, r2

rmse, r2 = make_direct_prediction(X_train, X_test, y_train, y_test, window_size=window_size, sliding_size=sliding_size, task=prediction_task)
print(f"Test RMSE: {rmse:.4f}, R2: {r2:.4f}")


In [ ]:
"cali_housing processing"
z_lin_dim, z_s_dim   = 18, 4
hidden_lin, hidden_s = 40, 25
cyc_features         = ["Latitude", "Longitude"]

def run_tabular_latent_housing(X_train: pd.DataFrame, X_test: pd.DataFrame, 
                               y_train: np.ndarray, y_test: np.ndarray, 
                               encoding: str = "cartesian"):
    """Encode tabular data into latent z and predict y. No windowing.
    encoding: 'cartesian' (3D coords), 'vmf' (spherical), 'gaussian' (euclidean)"""

    # --- Split linear vs cyclic (lat/lon) ---
    lin_features = [f for f in X_train.columns if f not in cyc_features]

    X_lin_train, X_cyc_train = X_train[lin_features].values, X_train[cyc_features].values
    X_lin_test,  X_cyc_test  = X_test[lin_features].values,  X_test[cyc_features].values

    # --- Scale ---
    X_lin_train, X_lin_test = scale_train_and_test_sets(X_lin_train, X_lin_test)
    X_cyc_train, X_cyc_test = scale_train_and_test_sets(X_cyc_train, X_cyc_test)

    # --- Convert to tensors on device ---
    X_lin_train = torch.as_tensor(X_lin_train, dtype=torch.float32, device=device)
    X_lin_test  = torch.as_tensor(X_lin_test,  dtype=torch.float32, device=device)
    X_cyc_train = torch.as_tensor(X_cyc_train, dtype=torch.float32, device=device)
    X_cyc_test  = torch.as_tensor(X_cyc_test,  dtype=torch.float32, device=device)

    # --- Define encoders AFTER data is tensors ---
    lin_encoder = EuclidEncoder(1, X_lin_train.shape[1], z_lin_dim, hidden_lin).to(device)
    cyc_encoder = SphericalEncoder(1, X_cyc_train.shape[1], z_s_dim, hidden_s).to(device)

    # --- Forward pass ---
    if encoding == "cartesian":
        # just normalize lat/lon to [-1,1] range as 3D coords
        z_s_train   = torch.cat([X_cyc_train, torch.zeros(X_cyc_train.shape[0], 1, device=device)], dim=1)
        z_s_test    = torch.cat([X_cyc_test,  torch.zeros(X_cyc_test.shape[0], 1, device=device)], dim=1)
        z_lin_train = lin_encoder(X_lin_train)[0]
        z_lin_test  = lin_encoder(X_lin_test)[0]
    elif encoding == "vmf":
        mu_s_train, kappa_train = cyc_encoder(X_cyc_train)
        z_s_train = Reparam.sample_vmf(mu_s_train, kappa_train)
        mu_s_test, kappa_test = cyc_encoder(X_cyc_test)
        z_s_test  = Reparam.sample_vmf(mu_s_test, kappa_test)
        z_lin_train, _ = lin_encoder(X_lin_train)
        z_lin_test, _  = lin_encoder(X_lin_test)
    elif encoding == "gaussian":
        z_lin_train, logvar_lin  = lin_encoder(X_lin_train)
        z_lin_test,  logvar_test = lin_encoder(X_lin_test)
        z_s_train = cyc_encoder(X_cyc_train)[0]  # just mean
        z_s_test  = cyc_encoder(X_cyc_test)[0]
    else:
        raise ValueError("encoding must be cartesian / vmf / gaussian")

    # --- Combine latents ---
    Z_train = torch.cat([z_lin_train, z_s_train], dim=1).detach().cpu().numpy()
    Z_test  = torch.cat([z_lin_test,  z_s_test],  dim=1).detach().cpu().numpy()

    # --- Predict with CatBoost ---
    y_hat = fit_catboost_multi(Z_train, y_train, Z_test)
    rmse  = np.sqrt(mean_squared_error(y_test, y_hat))
    r2    = r2_score(y_test, y_hat)
    return rmse, r2

for enc in ["cartesian", "vmf", "gaussian"]:
    rmse, r2 = run_tabular_latent_housing(X_train, X_test, y_train, y_test, encoding=enc)
    print(f"{enc:8s} latent: RMSE={rmse:.4f}, R2={r2:.4f}")


In [ ]:
"FUNCTION: CYCLIC + LINEAR split"

@dataclass
class RunParams:
    window_size: int
    sliding_size: int
    z_dim_total: int
    hidden_dim: int
    lr_optimizer: float
    batch_size: int
    epochs: int
    cyclic_threshold: float
    earlystop_patience: int = 8
    lambda_recon: float = 1.0
    lambda_latent: float = 1e-3
    task: str = prediction_task  # forecast, nowcast, tabular

def run_split_vae_scenario_MLP(X_train: pd.DataFrame, X_test: pd.DataFrame, y_train: np.ndarray, y_test: np.ndarray,
                           p: RunParams) -> Tuple[float, float]:
    """Run one split-VAE experiment. Returns (rmse, r2)."""

    hidden_dim_split = int(p.hidden_dim / np.sqrt(2))

    # ---- split linear / cyclic BEFORE scaling ----
    X_lin_train, X_cyc_train = split_dataset_to_linear_and_cyclic(X_train, threshold=p.cyclic_threshold, verbose = False)
    X_lin_test = X_test[X_lin_train.columns]
    X_cyc_test = X_test[X_cyc_train.columns]

    # ---- scale ----
    y_train_scaled, y_test_scaled = scale_train_and_test_sets(y_train, y_test)
    X_lin_train, X_lin_test = scale_train_and_test_sets(X_lin_train, X_lin_test)
    X_cyc_train, X_cyc_test = scale_train_and_test_sets(X_cyc_train, X_cyc_test)

    # ---- latent dim split (by cyclicity proportion) ----
    z_dim_euclid  = p.z_dim_total * X_lin_train.shape[-1] // X_train.shape[-1]
    z_dim_spheric = p.z_dim_total - z_dim_euclid
    print(f"z_e={z_dim_euclid}, z_s={z_dim_spheric}, " f"cyc%={X_cyc_train.shape[1]/X_train.shape[1]:.2f}")

    # ~~~~~~~
    cyc_ratio = X_cyc_train.shape[1] / X_train.shape[1]

    # raw proportional split
    z_dim_spheric = int(p.z_dim_total * cyc_ratio)
    z_dim_euclid  = p.z_dim_total - z_dim_spheric

    # enforce sensible bounds
    Z_E_MIN, Z_E_MAX = 20, 28
    Z_S_MIN          = 8

    z_dim_euclid  = max(Z_E_MIN, min(z_dim_euclid, Z_E_MAX))
    z_dim_spheric = p.z_dim_total - z_dim_euclid

    if z_dim_spheric < Z_S_MIN:
        z_dim_spheric = Z_S_MIN
        z_dim_euclid  = p.z_dim_total - z_dim_spheric
    # ~~~~~~~

    # ---- windowing ----
    X_lin_train_w = Windowing.make_windows_from_X(X_lin_train, p.window_size, p.sliding_size).to(device)
    X_cyc_train_w = Windowing.make_windows_from_X(X_cyc_train, p.window_size, p.sliding_size).to(device)
    X_lin_test_w  = Windowing.make_windows_from_X(X_lin_test,  p.window_size, p.sliding_size).to(device)
    X_cyc_test_w  = Windowing.make_windows_from_X(X_cyc_test,  p.window_size, p.sliding_size).to(device)

    # lstm addition
    X_lin_train_w = X_lin_train_w.view(-1, p.window_size, X_lin_train.shape[-1]).to(device)
    X_cyc_train_w = X_cyc_train_w.view(-1, p.window_size, X_cyc_train.shape[-1]).to(device)
    X_lin_test_w  = X_lin_test_w.view(-1, p.window_size, X_lin_test.shape[-1]).to(device)
    X_cyc_test_w  = X_cyc_test_w.view(-1, p.window_size, X_cyc_test.shape[-1]).to(device)

    # ---- models ----
    encoder_e = EuclidEncoder(window_size=p.window_size, input_dim=X_lin_train.shape[1], z_dim=z_dim_euclid, hidden=hidden_dim_split,).to(device)
    encoder_s = SphericalEncoder(window_size=p.window_size, input_dim=X_cyc_train.shape[1], z_dim=z_dim_spheric, hidden=hidden_dim_split,).to(device)
    decoder   = Decoder(z_dim_total=p.z_dim_total, window_size=p.window_size, output_dim=X_train.shape[1], hidden=p.hidden_dim,).to(device)

    # encoder_e = LSTMEncoderEuclid(input_dim=X_lin_train.shape[-1], hidden_dim=hidden_dim_split, z_dim=z_dim_euclid).to(device)
    # encoder_s = LSTMSphericalEncoder(input_dim=X_cyc_train.shape[-1], hidden_dim=hidden_dim_split, z_dim=z_dim_spheric).to(device)
    # decoder   = LSTMDecoder(z_dim_total=p.z_dim_total, window_size=p.window_size, output_dim=X_train.shape[-1], hidden_dim=p.hidden_dim).to(device)

    optimizer = torch.optim.AdamW(list(encoder_e.parameters()) + list(encoder_s.parameters()) + list(decoder.parameters()),
                                  lr=p.lr_optimizer,)
    # ---- training ----
    train_loader = DataLoader(TensorDataset(X_lin_train_w, X_cyc_train_w), batch_size=p.batch_size, shuffle=False,)

    lambdas   = {"reconstr": p.lambda_recon,
                 "euc": p.lambda_latent / np.sqrt(z_dim_euclid),
                 "sph": p.lambda_latent / np.sqrt(z_dim_spheric),}
    best_loss = float("inf")
    counter   = 0

    for _ in range(p.epochs):
        epoch_loss = WithSplit.train_linear_and_cyclic_vaes_for_1_epoch_tabular(train_loader, encoder_e, encoder_s, decoder, optimizer, lambdas)
        best_loss, counter, stop = early_stop(epoch_loss, best_loss, counter, p.earlystop_patience)
        if stop:
            break

    # ---- encode ----
    Z_train = WithSplit.encode_tabular_dataset(X_lin_train_w, X_cyc_train_w, encoder_e, encoder_s, pooling=None)
    Z_train = pool_latents(Z_train, z_e_dim=z_dim_euclid)
    Z_test  = WithSplit.encode_tabular_dataset(X_lin_test_w, X_cyc_test_w, encoder_e, encoder_s, pooling=None)
    Z_test  = pool_latents(Z_test, z_e_dim=z_dim_euclid)

    # ---- regression ----
    y_train_win = Windowing.make_windows_from_y(y_train_scaled, p.window_size, p.sliding_size, task=p.task)
    y_test_win  = Windowing.make_windows_from_y(y_test_scaled, p.window_size, p.sliding_size, task=p.task)

    # print("X_train_w shape", X_lin_train_w.shape)
    # print("X_test_w shape", X_lin_test_w.shape)
    # print("y_train_win shape", y_train_win.shape)
    # print("Z_train shape", Z_train.shape)

    assert Z_train.shape[0] == y_train_win.shape[0]
    assert Z_test.shape[0]  == y_test_win.shape[0]

    y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy(),)
    rmse  = np.sqrt(mean_squared_error(y_test_win, y_hat))
    r2    = r2_score(y_test_win, y_hat)
    return rmse, r2

def run_split_vae_scenario_LSTM(X_train: pd.DataFrame, X_test: pd.DataFrame, y_train: np.ndarray,
                                y_test: np.ndarray, p: RunParams) -> Tuple[float, float]:
    """Run split-VAE with LSTM encoders and MLP decoder for timeseries."""

    # ---- split linear / cyclic BEFORE scaling ----
    X_lin_train, X_cyc_train = split_dataset_to_linear_and_cyclic(X_train, threshold=p.cyclic_threshold, verbose=False)
    X_lin_test = X_test[X_lin_train.columns]
    X_cyc_test = X_test[X_cyc_train.columns]

    # ---- scale ----
    y_train_scaled, y_test_scaled = scale_train_and_test_sets(y_train, y_test)
    X_lin_train, X_lin_test       = scale_train_and_test_sets(X_lin_train, X_lin_test)
    X_cyc_train, X_cyc_test       = scale_train_and_test_sets(X_cyc_train, X_cyc_test)

    # ---- latent dim split ----
    cyc_ratio     = X_cyc_train.shape[1] / X_train.shape[1]
    z_dim_spheric = max(8, int(p.z_dim_total * cyc_ratio))
    z_dim_euclid  = p.z_dim_total - z_dim_spheric
    z_dim_euclid  = max(20, min(z_dim_euclid, 28))
    z_dim_spheric = p.z_dim_total - z_dim_euclid
    print(f"z_e={z_dim_euclid}, z_s={z_dim_spheric}, cyc%={cyc_ratio:.2f}")

    # ---- windowing ----
    X_lin_train_w = Windowing.make_windows_from_X(X_lin_train, p.window_size, p.sliding_size)
    X_cyc_train_w = Windowing.make_windows_from_X(X_cyc_train, p.window_size, p.sliding_size)
    X_lin_test_w  = Windowing.make_windows_from_X(X_lin_test,  p.window_size, p.sliding_size)
    X_cyc_test_w  = Windowing.make_windows_from_X(X_cyc_test,  p.window_size, p.sliding_size)

    # ---- move to device ----
    X_lin_train_w = X_lin_train_w.to(device)
    X_cyc_train_w = X_cyc_train_w.to(device)
    X_lin_test_w  = X_lin_test_w.to(device)
    X_cyc_test_w  = X_cyc_test_w.to(device)

    # ---- models ----
    hidden_dim_split = int(p.hidden_dim / np.sqrt(2))
    encoder_e= LSTMEncoderEuclid(input_dim=X_lin_train.shape[-1], hidden_dim=hidden_dim_split, z_dim=z_dim_euclid).to(device)
    encoder_s= LSTMSphericalEncoder(input_dim=X_cyc_train.shape[-1], hidden_dim=hidden_dim_split, z_dim=z_dim_spheric).to(device)
    decoder  = MLPDecoder(z_dim_total=p.z_dim_total, window_size=p.window_size, output_dim=X_train.shape[-1], hidden_dim=p.hidden_dim).to(device)
    optimizer= torch.optim.AdamW(list(encoder_e.parameters()) + list(encoder_s.parameters()) + list(decoder.parameters()), lr=p.lr_optimizer)

    # ---- training loader ----
    train_loader = DataLoader(TensorDataset(X_lin_train_w, X_cyc_train_w), batch_size=p.batch_size, shuffle=False)

    lambdas = {"reconstr": p.lambda_recon,
               "euc": p.lambda_latent / np.sqrt(z_dim_euclid),
               "sph": p.lambda_latent / np.sqrt(z_dim_spheric)}

    best_loss = float("inf")
    counter   = 0
    for _ in range(p.epochs):
        epoch_loss = WithSplit.train_linear_and_cyclic_vaes_for_1_epoch_timeseries(train_loader, encoder_e, encoder_s, decoder,
                                                                                   optimizer, lambdas)
        best_loss, counter, stop = early_stop(epoch_loss, best_loss, counter, p.earlystop_patience)
        if stop:
            break

    # ---- encode ----
    Z_train = WithSplit.encode_timeseries_dataset(X_lin_train_w, X_cyc_train_w, encoder_e, encoder_s, pooling=None)
    Z_train = pool_latents(Z_train, z_e_dim=z_dim_euclid)
    Z_test  = WithSplit.encode_timeseries_dataset(X_lin_test_w, X_cyc_test_w, encoder_e, encoder_s, pooling=None)
    Z_test  = pool_latents(Z_test, z_e_dim=z_dim_euclid)

    # ---- regression ----
    y_train_win = Windowing.make_windows_from_y(y_train_scaled, p.window_size, p.sliding_size, task=p.task)
    y_test_win  = Windowing.make_windows_from_y(y_test_scaled, p.window_size, p.sliding_size, task=p.task)

    assert Z_train.shape[0] == y_train_win.shape[0]
    assert Z_test.shape[0]  == y_test_win.shape[0]

    y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy())
    rmse  = np.sqrt(mean_squared_error(y_test_win, y_hat))
    r2    = r2_score(y_test_win, y_hat)
    return rmse, r2


p = RunParams(
    z_dim_total=random.choice([20, 56, 60]),
    hidden_dim=random.choice([36, 40, 44]),
    window_size= window_size,#random.choice([32]),
    sliding_size=sliding_size,#random.choice([8]),
    lr_optimizer=10 ** random.uniform(-3.45, -3.35),
    batch_size=random.choice([256]),
    epochs=epochs,
    cyclic_threshold=random.choice([0.25, 0.26, 0.27, 0.28]),
    lambda_recon=random.uniform(1.58, 1.62),
    lambda_latent=random.uniform(0.0014, 0.002),)

if prediction_task == "tabular":
    rmse, r2 = run_split_vae_scenario_MLP(X_train, X_test, y_train, y_test, p)
else:
    rmse, r2 = run_split_vae_scenario_LSTM(X_train, X_test, y_train, y_test, p)
print(f"RMSE ({prediction_task}): {rmse:.4f} R2: {r2:.4f}")


In [ ]:
"hyperparam run for CYCLIC + LINEAR split"

results    = []
best_rmse  = float("inf")
best_params= None

num_trials = 50
for i in range(num_trials):
    p = RunParams(
        z_dim_total=random.choice([56, 60]),
        hidden_dim=random.choice([42, 44, 46]),
        window_size= window_size,#random.choice([32]),
        sliding_size=sliding_size,#random.choice([8]),
        lr_optimizer=10 ** random.uniform(-3.45, -3.35),
        batch_size=random.choice([128, 256, 512]),
        epochs=epochs,
        cyclic_threshold=random.choice([0.25, 0.3, 0.35, 0.4]),
        lambda_recon=random.uniform(1.603, 1.615),
        lambda_latent=random.uniform(0.0016, 0.0018),)

    rmse, r2 = run_split_vae_scenario_LSTM(X_train, X_test, y_train, y_test, p)
    results.append({**p.__dict__, "rmse": rmse, "r2": r2})

    if rmse < best_rmse:
        best_rmse   = rmse
        best_params = p
        print(f"New best RMSE: {best_rmse:.4f} at run #{i}")

    print(f"run #{i} RMSE: {rmse:.4f} R2: {r2:.4f}, "
        f"lambda_recon={p.lambda_recon:.4f}, lambda_latent={p.lambda_latent:.4f}, params: {p}")

results_df = pd.DataFrame(results).sort_values("rmse")
results_df.head()
results_df.to_csv("vae_param_sweep.csv", index=False, mode= 'a')


In [ ]:
"[to remove, just check that all functionality was done in the new version] CYCLIC + LINEAR split"

# 2. split to linear/cyclic BEFORE scaling, as scaling changes the cyclicity measure
X_lin_train, X_cyc_train = split_dataset_to_linear_and_cyclic(X_train, threshold=cyclic_threshold)

# select same cols as X_train
X_lin_test    = X_test[X_lin_train.columns]
X_cyc_test    = X_test[X_cyc_train.columns]

# 3. scale linear and cyclic SEPARATELY

y_train_scaled, y_test_scaled= scale_train_and_test_sets(y_train, y_test)
X_lin_train, X_lin_test = scale_train_and_test_sets(X_lin_train, X_lin_test)
X_cyc_train, X_cyc_test = scale_train_and_test_sets(X_cyc_train, X_cyc_test)

# latent estimate after scaling so distributions are consistent, after test/train split so no data leakage
z_dim_euclid  = z_dim_total * X_lin_train.shape[-1] // X_train.shape[-1]
z_dim_spheric = z_dim_total - z_dim_euclid
# H_lin = sum(estimate_entropy(X_lin_train[:, i])
#             for i in range(X_lin_train.shape[1]))
# H_cyc = sum(estimate_entropy(X_cyc_train[:, i])
#             for i in range(X_cyc_train.shape[1]))
# z_dim_euclid  = int(z_dim_total * H_lin / (H_lin + H_cyc))
# z_dim_spheric = z_dim_total - z_dim_euclid
# print(f"H_lin: {H_lin:.4f}= {H_lin/(H_lin+H_cyc):.4f}, H_cyc: {H_cyc:.4f}= {H_cyc/(H_lin+H_cyc):.4f}")
print(f"z_e: {z_dim_euclid}, z_s: {z_dim_spheric}")

# window data AFTER splitting to linear/cyclic, as the sliding window would mess a feature's cyclicity measure if done before
X_lin_train_w = Windowing.make_windows_from_X(X_lin_train, window_size).to(device)
X_cyc_train_w = Windowing.make_windows_from_X(X_cyc_train, window_size).to(device)
X_lin_test_w  = Windowing.make_windows_from_X(X_lin_test, window_size).to(device)
X_cyc_test_w  = Windowing.make_windows_from_X(X_cyc_test, window_size).to(device)
X_train_w     = torch.cat([X_lin_train_w, X_cyc_train_w], dim=-1).to(device)
X_test_w      = torch.cat([X_lin_test_w, X_cyc_test_w], dim=-1).to(device)

encoder_e = EuclidEncoder(window_size=window_size, input_dim=X_lin_train.shape[1], z_dim=z_dim_euclid, hidden=hidden_dim_split).to(device)
encoder_s = SphericalEncoder(window_size=window_size, input_dim=X_cyc_train.shape[1], z_dim=z_dim_spheric, hidden=hidden_dim_split).to(device)
decoder   = Decoder(z_dim_total=z_dim_euclid + z_dim_spheric, window_size=window_size, output_dim=X_train.shape[1], hidden=hidden_dim).to(device)
optimizer = torch.optim.AdamW(list(encoder_e.parameters()) + list(encoder_s.parameters())+ list(decoder.parameters()), lr=lr_optimizer)

# training
train_ds     = TensorDataset(X_lin_train_w, X_cyc_train_w)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)

lambdas    = {"reconstr": 1.0, "euc": 1e-3/z_dim_euclid, "sph": 1e-3/z_dim_spheric}

best_loss  = float('inf')
counter    = 0
time_start = time.time()
for epoch in range(epochs):
    epoch_loss = WithSplit.train_linear_and_cyclic_vaes_for_1_epoch_timeseries(train_loader, encoder_e, encoder_s, decoder,
                                                                               optimizer, lambdas)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}")
    best_loss, counter, stop = early_stop(epoch_loss, best_loss, counter, earlystop_patience)
    if stop:
        print(f"Stopping early at epoch {epoch+1}")
        break
time_end = time.time()
print(f"Training time: {time_end - time_start:.2f}s")

# encode without collapsing all windows
Z_train = WithSplit.encode_tabular_dataset(X_lin_train_w, X_cyc_train_w, encoder_e, encoder_s, pooling=None)
Z_train = pool_latents(Z_train, z_e_dim=z_dim_euclid)
Z_test  = WithSplit.encode_tabular_dataset(X_lin_test_w, X_cyc_test_w, encoder_e, encoder_s, pooling=None)
Z_test  = pool_latents(Z_test, z_e_dim=z_dim_euclid)

# Cut y_train/y_test to match number of windows
y_train_win = y_train_scaled[:Z_train.shape[0]]
y_test_win  = y_test_scaled[:Z_test.shape[0]]

y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy())
rmse  = np.sqrt(mean_squared_error(y_test_win, y_hat))
r2    = r2_score(y_test_win, y_hat)
print(f"Test RMSE: {rmse:.4f}, R2: {r2:.4f}")


In [ ]:
"NEW split"
# ---------------------------
# 1. split linear vs cyclic features BEFORE scaling
# ---------------------------
X_lin_train, X_cyc_train = split_dataset_to_linear_and_cyclic(X_train, threshold=cyclic_threshold)
X_lin_test   = X_test[X_lin_train.columns]
X_cyc_test   = X_test[X_cyc_train.columns]

# ---------------------------
# 2. scale separately
# ---------------------------
y_scaler       = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled  = y_scaler.transform(y_test)

X_lin_scaler   = StandardScaler()
X_lin_train    = pd.DataFrame(X_lin_scaler.fit_transform(X_lin_train), columns=X_lin_train.columns)
X_lin_test     = pd.DataFrame(X_lin_scaler.transform(X_lin_test), columns=X_lin_train.columns)

X_cyc_scaler   = StandardScaler()
X_cyc_train    = pd.DataFrame(X_cyc_scaler.fit_transform(X_cyc_train), columns=X_cyc_train.columns)
X_cyc_test     = pd.DataFrame(X_cyc_scaler.transform(X_cyc_test), columns=X_cyc_train.columns)

X_lin_train  = torch.tensor(X_lin_train.values, dtype=torch.float32)
X_lin_test   = torch.tensor(X_lin_test.values, dtype=torch.float32)
X_cyc_train  = torch.tensor(X_cyc_train.values, dtype=torch.float32)
X_cyc_test   = torch.tensor(X_cyc_test.values, dtype=torch.float32)

# ---------------------------
# 3. latent dimensions split
# ---------------------------
z_dim_euclid  = z_dim_total * X_lin_train.shape[-1] // X_train.shape[-1]
z_dim_spheric = z_dim_total - z_dim_euclid  # ensure sum matches decoder input
print(f"Latent dims -> Euclidean: {z_dim_euclid}, Spherical: {z_dim_spheric}")

# ---------------------------
# 4. window the data
# ---------------------------
X_lin_train_w = Windowing.make_windows_from_X(X_lin_train, window_size).to(device)
X_cyc_train_w = Windowing.make_windows_from_X(X_cyc_train, window_size).to(device)
X_lin_test_w  = Windowing.make_windows_from_X(X_lin_test, window_size).to(device)
X_cyc_test_w  = Windowing.make_windows_from_X(X_cyc_test, window_size).to(device)
X_train_w     = torch.cat([X_lin_train_w, X_cyc_train_w], dim=-1).to(device)
X_test_w      = torch.cat([X_lin_test_w, X_cyc_test_w], dim=-1).to(device)

# ---------------------------
# 5. Shared Encoder / Decoder Setup
# input_dim for encoder is the TOTAL features (lin + cyc)
total_input_dim = X_train.shape[1] 
encoder_mixed = MixedEncoder(input_dim=total_input_dim * window_size, 
                             hidden_dim=hidden_dim_split, 
                             z_e_dim=z_dim_euclid, 
                             z_s_dim=z_dim_spheric).to(device)

decoder = Decoder(z_dim_total=z_dim_euclid + z_dim_spheric, 
                  window_size=window_size, 
                  output_dim=total_input_dim, 
                  hidden=hidden_dim).to(device)

optimizer = torch.optim.AdamW(list(encoder_mixed.parameters()) + list(decoder.parameters()), lr=lr_optimizer)

# 6. Modified Training Step for Specialization
def train_step_specialized(x_full_w, lin_idx, cyc_idx, encoder, decoder, lambdas):
    B, win, D = x_full_w.shape
    x_flat = x_full_w.view(B, -1)
    
    # Encoder head outputs
    mu_e, logvar_e, mu_s, kappa = encoder(x_flat)
    
    # Reparameterize
    z_e = Reparam.reparam_gaussian(mu_e, logvar_e)
    z_s = Reparam.reparam_vmf(mu_s, kappa)
    z = torch.cat([z_e, z_s], dim=-1)
    
    # Decode
    x_hat_flat = decoder(z)
    x_hat = x_hat_flat.view(B, win, D)
    
    # Specialization Loss: Gradients for z_e come from lin, z_s from cyc
    # We slice the original and reconstructed tensors
    loss_recon_lin = F.mse_loss(x_hat[:, :, lin_idx], x_full_w[:, :, lin_idx])
    loss_recon_cyc = F.mse_loss(x_hat[:, :, cyc_idx], x_full_w[:, :, cyc_idx])
    
    L_kl_e = kl_gaussian(mu_e, logvar_e)
    L_kl_s = kl_vmf_uniform(mu_s, kappa)
    
    return lambdas["reconstr"] * (loss_recon_lin + loss_recon_cyc) + \
           lambdas["euc"] * L_kl_e + lambdas["sph"] * L_kl_s

# 7. Training Loop
# Identify indices for slicing
lin_indices = torch.arange(X_lin_train.shape[1])
cyc_indices = torch.arange(X_lin_train.shape[1], total_input_dim)

train_loader = DataLoader(TensorDataset(X_train_w), batch_size=batch_size, shuffle=True)

time_start = time.time()
for epoch in range(epochs):
    encoder_mixed.train(); decoder.train()
    epoch_loss = 0
    for (batch_x,) in train_loader:
        optimizer.zero_grad()
        loss = train_step_specialized(batch_x, lin_indices, cyc_indices, encoder_mixed, decoder, lambdas)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {epoch_loss/len(train_loader):.4f}")
time_end = time.time()
print(f"Training time: {time_end - time_start:.2f}s")

# 8. Encoding & Latent Pooling
encoder_mixed.eval()
with torch.no_grad():
    # Process train
    mu_e, _, mu_s, _ = encoder_mixed(X_train_w.view(X_train_w.size(0), -1))
    Z_train = torch.cat([mu_e, mu_s], dim=-1).unsqueeze(1) # shape (N, 1, Z)
    # Since MixedEncoder doesn't naturally output window sequences here, 
    # we treat it as 1 window or use pool_latents if you modify forward to handle windows.
    Z_train = pool_latents(Z_train, z_e_dim=z_dim_euclid)

    # Process test
    mu_e_t, _, mu_s_t, _ = encoder_mixed(X_test_w.view(X_test_w.size(0), -1))
    Z_test = torch.cat([mu_e_t, mu_s_t], dim=-1).unsqueeze(1)
    Z_test = pool_latents(Z_test, z_e_dim=z_dim_euclid)

# 9. Regression (Keep same)
y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_scaled[:Z_train.shape[0]], Z_test.cpu().numpy())
rmse  = np.sqrt(mean_squared_error(y_test_win, y_hat))
r2    = r2_score(y_test_win, y_hat)
print(f"Test RMSE: {rmse:.4f}, R2: {r2:.4f}")


In [ ]:
"100% EUCLIDEAN (NO SPLIT)"

def run_euclidean_vae_scenario_MLP(X_train: pd.DataFrame, X_test: pd.DataFrame, y_train: np.ndarray,
                                   y_test: np.ndarray, p: RunParams,) -> Tuple[float, float]:
    """Run Euclidean-only VAE baseline. Returns (rmse, r2)."""

    hidden_dim_split = int(p.hidden_dim / np.sqrt(2))

    # ---- scale (match split-VAE MLP) ----
    X_train, X_test = scale_train_and_test_sets(X_train, X_test)
    y_train, y_test = scale_train_and_test_sets(y_train, y_test)

    # ---- windowing ----
    X_train_w = Windowing.make_windows_from_X(X_train, p.window_size, p.sliding_size).to(device)
    X_test_w  = Windowing.make_windows_from_X(X_test, p.window_size, p.sliding_size).to(device)

    X_train_w = X_train_w.view(-1, p.window_size, X_train.shape[-1]).to(device)
    X_test_w  = X_test_w.view(-1, p.window_size, X_test.shape[-1]).to(device)

    # ---- model ----
    encoder   = EuclidEncoder(window_size=p.window_size, input_dim=X_train.shape[1], z_dim=p.z_dim_total, hidden=hidden_dim_split).to(device)
    decoder   = Decoder(z_dim_total=p.z_dim_total, window_size=p.window_size, output_dim=X_train.shape[1], hidden=hidden_dim_split).to(device)
    optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=p.lr_optimizer,)

    # ---- training ----
    train_loader = DataLoader(TensorDataset(X_train_w), batch_size=p.batch_size, shuffle=False,)

    lambdas = {
        "reconstr": p.lambda_recon,
        "euc": p.lambda_latent / np.sqrt(p.z_dim_total),
        "sph": 0.0,}

    best_loss = float("inf")
    counter   = 0
    for _ in range(p.epochs):
        epoch_loss = NoSplit.train_vae_no_split(train_loader, encoder, decoder, optimizer, lambdas)
        best_loss, counter, stop = early_stop(epoch_loss, best_loss, counter, p.earlystop_patience)
        if stop:
            break

    # ---- encode ----
    Z_train = NoSplit.encode_dataset_no_split(X_train_w, encoder, pooling=None)
    Z_test  = NoSplit.encode_dataset_no_split(X_test_w, encoder, pooling=None)

    # ---- regression ----
    y_train_win = Windowing.make_windows_from_y(y_train_scaled, p.window_size, p.sliding_size, task=p.task)
    y_test_win  = Windowing.make_windows_from_y(y_test_scaled, p.window_size, p.sliding_size, task=p.task)

    assert Z_train.shape[0] == y_train_win.shape[0]
    assert Z_test.shape[0]  == y_test_win.shape[0]

    y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy(),)
    rmse  = np.sqrt(mean_squared_error(y_test_win, y_hat))
    r2    = r2_score(y_test_win, y_hat)
    return rmse, r2

def run_euclidean_vae_scenario_LSTM(X_train: pd.DataFrame, X_test: pd.DataFrame, y_train: np.ndarray,
                                    y_test: np.ndarray, p: RunParams) -> tuple[float, float]:
    """Run Euclidean VAE with LSTM encoder and MLP decoder."""

    hidden_dim_split = int(p.hidden_dim / np.sqrt(2))

    # ---- scale ----
    X_train_t, X_test_t = scale_train_and_test_sets(X_train, X_test)
    y_train_scaled, y_test_scaled = scale_train_and_test_sets(y_train, y_test)

    # ---- windowing ----
    X_train_w = Windowing.make_windows_from_X(X_train_t, p.window_size, p.sliding_size).to(device)
    X_test_w  = Windowing.make_windows_from_X(X_test_t, p.window_size, p.sliding_size).to(device)

    # ---- models ----
    encoder   = LSTMEncoderEuclid(input_dim=X_train.shape[1], hidden_dim=hidden_dim_split, z_dim=p.z_dim_total).to(device)
    decoder   = MLPDecoder(z_dim_total=p.z_dim_total, window_size=p.window_size, output_dim=X_train.shape[1], hidden_dim=hidden_dim_split,).to(device)
    optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=p.lr_optimizer,)

    # ---- training ----
    train_loader = DataLoader(TensorDataset(X_train_w), batch_size=p.batch_size, shuffle=False,)

    lambdas = {"reconstr": p.lambda_recon, "euc": p.lambda_latent / np.sqrt(p.z_dim_total),}

    best_loss = float("inf")
    counter   = 0
    for _ in range(p.epochs):
        epoch_loss = 0.0
        for (x,) in train_loader:
            optimizer.zero_grad()

            mu, logvar = encoder(x)
            z          = Reparam.reparam_gaussian(mu, logvar)

            x_hat_flat = decoder(z)
            x_hat      = x_hat_flat.view_as(x)

            L_rec = F.mse_loss(x_hat, x)
            L_kl  = kl_gaussian(mu, logvar)
            loss  = lambdas["reconstr"] * L_rec + lambdas["euc"] * L_kl
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        best_loss, counter, stop = early_stop(epoch_loss, best_loss, counter, p.earlystop_patience)
        if stop:
            break

    # ---- encode (NO POOLING) ----
    encoder.eval()
    with torch.no_grad():
        mu_train, _ = encoder(X_train_w)
        mu_test,  _ = encoder(X_test_w)

    Z_train = mu_train
    Z_test  = mu_test

    # ---- regression ----
    y_train_win = Windowing.make_windows_from_y(y_train_scaled, p.window_size, p.sliding_size, task=p.task)
    y_test_win  = Windowing.make_windows_from_y(y_test_scaled, p.window_size, p.sliding_size, task=p.task)

    assert Z_train.shape[0] == y_train_win.shape[0]
    assert Z_test.shape[0]  == y_test_win.shape[0]

    y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy(),)
    rmse  = np.sqrt(mean_squared_error(y_test_win, y_hat))
    r2    = r2_score(y_test_win, y_hat)
    return rmse, r2

if prediction_task == "tabular":
    rmse, r2 = run_euclidean_vae_scenario_MLP(X_train, X_test, y_train, y_test, p)
else:
    rmse, r2 = run_euclidean_vae_scenario_LSTM(X_train, X_test, y_train, y_test, p)
print(f"RMSE ({prediction_task}): {rmse:.4f} R2: {r2:.4f}")



In [ ]:
"100% SPHERICAL"

def run_spherical_vae_scenario_MLP(X_train: pd.DataFrame,X_test: pd.DataFrame, y_train: np.ndarray,
                                   y_test: np.ndarray, p: RunParams,) -> tuple[float, float]:
    """Run 100% spherical VAE baseline. Returns (rmse, r2)."""

    hidden_dim_split = int(p.hidden_dim / np.sqrt(2))

    # ---- scale (match Euclidean baseline) ----
    X_train, X_test = scale_train_and_test_sets(X_train, X_test)
    y_train_scaled, y_test_scaled = scale_train_and_test_sets(y_train, y_test)

    # ---- windowing ----
    X_train_w = Windowing.make_windows_from_X(X_train, p.window_size, p.sliding_size).to(device)
    X_test_w  = Windowing.make_windows_from_X(X_test, p.window_size, p.sliding_size).to(device)

    X_train_w = X_train_w.view(-1, p.window_size, X_train.shape[-1]).to(device)
    X_test_w  = X_test_w.view(-1, p.window_size, X_test.shape[-1]).to(device)

    # ---- model ----
    encoder  = SphericalEncoder(window_size=p.window_size, input_dim=X_train.shape[1], z_dim=p.z_dim_total, hidden=hidden_dim_split).to(device)
    decoder  = Decoder(z_dim_total=p.z_dim_total,window_size=p.window_size, output_dim=X_train.shape[1],hidden=hidden_dim_split).to(device)
    optimizer= torch.optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()),lr=p.lr_optimizer,)

    # ---- training ----
    train_loader = DataLoader(TensorDataset(X_train_w), batch_size=p.batch_size, shuffle=False)

    lambdas = {
        "reconstr": p.lambda_recon,
        "sph": p.lambda_latent,
        "euc": 0.0,}

    best_loss = float("inf")
    counter   = 0

    for _ in range(p.epochs):
        epoch_loss = 0.0
        encoder.train()
        decoder.train()

        for (x,) in train_loader:
            optimizer.zero_grad()

            B, win, D = x.shape
            x_flat    = x.view(B, win * D)
            mu, kappa = encoder(x_flat)
            z         = Reparam.reparam_vmf(mu, kappa)

            x_hat     = decoder(z)
            L_rec     = F.mse_loss(x_hat, x_flat)
            L_sph     = regularization_vmf(kappa, z.size(-1))

            loss = lambdas["reconstr"] * L_rec + lambdas["sph"] * L_sph
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        best_loss, counter, stop = early_stop(epoch_loss, best_loss, counter, p.earlystop_patience)
        if stop:
            break

    # ---- encode (NO EUCLIDEAN MEAN) ----
    encoder.eval()
    with torch.no_grad():
        mu_train, _ = encoder(X_train_w.view(X_train_w.size(0), -1))
        mu_test,  _ = encoder(X_test_w.view(X_test_w.size(0), -1))

    Z_train = F.normalize(mu_train, dim=-1)
    Z_test  = F.normalize(mu_test,  dim=-1)

    # ---- regression ----
    y_train_win = Windowing.make_windows_from_y(y_train_scaled, p.window_size, p.sliding_size, task=p.task)
    y_test_win  = Windowing.make_windows_from_y(y_test_scaled, p.window_size, p.sliding_size, task=p.task)

    assert Z_train.shape[0] == y_train_win.shape[0]
    assert Z_test.shape[0]  == y_test_win.shape[0]

    y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy(),)
    rmse  = np.sqrt(mean_squared_error(y_test_win, y_hat))
    r2    = r2_score(y_test_win, y_hat)
    return rmse, r2

def run_spherical_vae_scenario_LSTM(X_train: pd.DataFrame, X_test: pd.DataFrame, y_train: np.ndarray,
                                    y_test: np.ndarray, p: RunParams,) -> tuple[float, float]:
    """Run 100% spherical VAE with LSTM encoder and MLP decoder for timeseries."""

    hidden_dim_split = int(p.hidden_dim / np.sqrt(2))

    # ---- scale ----
    X_train_scaled, X_test_scaled = scale_train_and_test_sets(X_train, X_test)
    y_train_scaled, y_test_scaled = scale_train_and_test_sets(y_train, y_test)

    # ---- windowing ----
    X_train_w = Windowing.make_windows_from_X(X_train_scaled, p.window_size, p.sliding_size).to(device)
    X_test_w  = Windowing.make_windows_from_X(X_test_scaled,  p.window_size, p.sliding_size).to(device)

    # ---- reshape for LSTM: (B, T, D) ----
    X_train_w = X_train_w.view(-1, p.window_size, X_train.shape[-1]).to(device)
    X_test_w  = X_test_w.view(-1, p.window_size, X_test.shape[-1]).to(device)

    # ---- model ----
    encoder   = LSTMSphericalEncoder(input_dim=X_train.shape[1], hidden_dim=hidden_dim_split, z_dim=p.z_dim_total).to(device)
    decoder   = MLPDecoder(z_dim_total=p.z_dim_total, window_size=p.window_size, output_dim=X_train.shape[1], hidden_dim=hidden_dim_split ).to(device)
    optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=p.lr_optimizer)

    # ---- training ----
    train_loader = DataLoader(TensorDataset(X_train_w), batch_size=p.batch_size, shuffle=False)

    lambdas = {"reconstr": p.lambda_recon, "sph": p.lambda_latent, "euc": 0.0}

    best_loss = float("inf")
    counter   = 0

    for _ in range(p.epochs):
        epoch_loss = 0.0
        encoder.train()
        decoder.train()

        for (x,) in train_loader:
            optimizer.zero_grad()

            mu, kappa = encoder(x)              # LSTM encoder
            z         = Reparam.reparam_vmf(mu, kappa)

            x_hat_flat = decoder(z)
            x_hat      = x_hat_flat.view_as(x)

            L_rec = F.mse_loss(x_hat, x)
            L_sph = regularization_vmf(kappa, z.size(-1))
            loss  = lambdas["reconstr"] * L_rec + lambdas["sph"] * L_sph
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        best_loss, counter, stop = early_stop(epoch_loss, best_loss, counter, p.earlystop_patience)
        if stop:
            break

    # ---- encode (no Euclidean mean) ----
    encoder.eval()
    with torch.no_grad():
        mu_train, _ = encoder(X_train_w)
        mu_test,  _ = encoder(X_test_w)

    Z_train = F.normalize(mu_train, dim=-1)
    Z_test  = F.normalize(mu_test,  dim=-1)

    # ---- regression ----
    y_train_win = Windowing.make_windows_from_y(y_train_scaled, p.window_size, p.sliding_size, task=p.task)
    y_test_win  = Windowing.make_windows_from_y(y_test_scaled,  p.window_size, p.sliding_size, task=p.task)

    assert Z_train.shape[0] == y_train_win.shape[0]
    assert Z_test.shape[0]  == y_test_win.shape[0]

    y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy(),)
    rmse  = np.sqrt(mean_squared_error(y_test_win, y_hat))
    r2    = r2_score(y_test_win, y_hat)
    return rmse, r2


if prediction_task == "tabular":
    rmse, r2 = run_euclidean_vae_scenario_MLP(X_train, X_test, y_train, y_test, p)
else:
    rmse, r2 = run_euclidean_vae_scenario_LSTM(X_train, X_test, y_train, y_test, p)
print(f"RMSE ({prediction_task}): {rmse:.4f} R2: {r2:.4f}")



In [ ]:
"Ts2vec"
from src.ts2vec_model.ts2vec import TS2Vec # the real ts2vec library
from src.encoders.ts2vec_encoder import TS2VecEncoder
from src.utils.metrics_utils import Preds

# below uses the original ts2vec (class from see ts2vec_encoder.py)
class TS2VecEncoder:
    """TS2Vec + Linear Predictor pipeline with externally set hyperparameters
       Supports early stopping during TS2Vec training."""

    def __init__(self, lr, z_pooling: str = "mean", device=None, patience: int = 20, max_train_length: int = 512):
        """Args:
            z_pooling (str): Pooling method for encoding.
            device (str or torch.device): "cpu" or "cuda".
            patience (int): Number of epochs with no improvement before stopping"""
        self.z_pooling   = z_pooling
        self.ts_model    = None
        self.device      = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.patience    = patience
        self._stop_early = False  # flag to indicate if early stopping occurred
        self.lr          = lr
        self.max_train_length = max_train_length

    class EarlyStopper:
        """Callback for TS2Vec early stopping."""
        def __init__(self, patience: int):
            self.patience = patience
            self.best = float("inf")
            self.wait = 0
            self.stop = False

        def __call__(self, model, loss: float):
            if loss < self.best:
                self.best = loss
                self.wait = 0
            else:
                self.wait += 1
                if self.wait >= self.patience:
                    print(f"Early stopping at epoch {model.n_epochs}")
                    self.stop = True
                    model.n_epochs = 1e9  # forces loop exit

    def fit_ts2vec(self, X_train, hidden_dims=12, output_dims=8, depth=5, batch_size=16, n_epochs=20):
        """Fit TS2Vec on training data with early stopping."""
        stopper       = self.EarlyStopper(self.patience)
        self.ts_model = TS2Vec(input_dims=X_train.shape[2], hidden_dims=hidden_dims, depth=depth, lr=self.lr,
                               output_dims=output_dims, batch_size=batch_size, device=self.device,
                               after_epoch_callback=stopper, max_train_length=self.max_train_length, temporal_unit=1)
        # self.ts_model.fit(X_train.astype(np.float32), n_epochs=n_epochs, verbose=True)
        X_np = X_train.cpu().numpy() if torch.is_tensor(X_train) else X_train
        self.ts_model.fit(X_np.astype(np.float32), n_epochs=n_epochs, verbose=True)


    def encode(self, X, pooling: str | None = "mean") -> np.ndarray:
        """Encode input X using trained TS2Vec model.
        Args:
            X: np.ndarray of shape (N, T, D)
            pooling: "mean" or None
        Returns:
            z: np.ndarray of shape (N, latent_dim) if pooled, else (N, T, latent_dim)"""
        if self.ts_model is None:
            raise ValueError("TS2Vec model not trained yet. Call fit_ts2vec first.")

        X_np = X.astype(np.float32) if isinstance(X, np.ndarray) else X.cpu().numpy()
        z    = self.ts_model.encode(X_np)  # (N, T, latent_dim)

        if pooling == "mean":
            return z.mean(axis=1)  # pool over time
        if pooling is None:
            return z
        raise ValueError(f"Unknown pooling: {pooling}")


# from ts2vec_runner.py, runs TS2VecEncoder class 
def run_ts2vec(X_train, X_test, y_train_scaled, y_test_scaled, *,
               model_cfg: dict, train_cfg: dict, device, save_ts2vec_encoder=False,):
    """Train TS2Vec with config dicts. Returns (losses, profiling_metrics)."""

    ts2vec = TS2VecEncoder(
        z_pooling= model_cfg["z_pooling_method"],
        lr       = train_cfg["lr"],
        device   = device,
        patience = train_cfg["patience"],
        max_train_length=train_cfg["window_size"],)

    metrics = ts2vec.fit_ts2vec(X_train,
        hidden_dims= model_cfg["hidden_dims"],
        output_dims= model_cfg["latent_dims"],
        depth      = model_cfg["depth"],
        batch_size = train_cfg["batch_size"],
        n_epochs   = train_cfg["epochs"],)

    z_train = ts2vec.encode(X_train, pooling=None)
    z_test  = ts2vec.encode(X_test,  pooling=None)

    print(f"{z_train.shape=}, {z_test.shape=}")

    z_train_flat = z_train.reshape(len(z_train), -1)
    z_test_flat  = z_test.reshape(len(z_test),  -1)

    print(f"{z_train_flat.shape=}, {z_test_flat.shape=}")

    losses, rf_model= Preds().evaluate_models_on_dataset(z_train_flat, y_train_scaled, z_test_flat,  y_test_scaled)
    r2              = rf_model.score(z_test_flat, y_test_scaled)
    return losses, r2, metrics


# --- configs ---
model_cfg = {
    "z_pooling_method": "mean",  # how TS2Vec pools over time
    "hidden_dims": 64,
    "latent_dims": 16,
    "depth": 5,}

train_cfg = {
    "lr": 1e-3,
    "patience": 20,
    "window_size": 128,  # max sequence length for TS2Vec
    "batch_size": 16,
    "epochs": 20,}

# --- scale ---
X_train_t, X_test_t           = scale_train_and_test_sets(X_train, X_test)
y_train_scaled, y_test_scaled = scale_train_and_test_sets(y_train, y_test)

# --- convert to numpy and ensure float32 ---
X_train_np = X_train_t.cpu().numpy() if torch.is_tensor(X_train_t) else np.array(X_train_t, dtype=np.float32)
X_test_np  = X_test_t.cpu().numpy()  if torch.is_tensor(X_test_t)  else np.array(X_test_t,  dtype=np.float32)

# --- make sliding windows ---
X_train_win = Windowing.make_windows_from_X(X_train_np, window_size, sliding_size)  # (num_windows, window_size, D)
X_test_win  = Windowing.make_windows_from_X(X_test_np,  window_size, sliding_size)
y_train_win = Windowing.make_windows_from_y(y_train_scaled, window_size, sliding_size, prediction_task)
y_test_win  = Windowing.make_windows_from_y(y_test_scaled,  window_size, sliding_size, prediction_task)

print(f"X_train_win: {X_train_win.shape}, y_train_win: {y_train_win.shape}")
print(f"X_test_win:  {X_test_win.shape}, y_test_win:  {y_test_win.shape}")

# --- run TS2Vec ---
losses, r2, _ = run_ts2vec(X_train_win, X_test_win, y_train_win, y_test_win,
                           model_cfg=model_cfg, train_cfg=train_cfg, device=device)
print(f"Simple TS2Vec: Losses={losses}, R2={r2:.4f}")

# --- encode to DataFrame safely ---
z_train_flat, z_test_flat = (
    np.array(X_train_win).reshape(len(X_train_win), -1),
    np.array(X_test_win).reshape(len(X_test_win), -1))

Z_train_df = pd.DataFrame(z_train_flat)
Z_test_df  = pd.DataFrame(z_test_flat)

# --- feed into hybrid split-VAE ---
rmse, r2_hybrid = run_split_vae_scenario_LSTM(Z_train_df, Z_test_df, y_train_win, y_test_win, p)
print(f"Hybrid VAE after TS2Vec: RMSE={rmse:.4f}, R2={r2_hybrid:.4f}")


In [ ]:
"ts2vec part 2"

def run_ts2vec_plain(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: np.ndarray,
    y_test: np.ndarray,
    p: RunParams,
    model_cfg: dict,
    train_cfg: dict,
    device):
    """Baseline: full X → TS2Vec → regress (no geometry)"""

    # -------------------------
    # 0. Scale
    # -------------------------
    y_train_s, y_test_s = scale_train_and_test_sets(y_train, y_test)
    X_train, X_test     = scale_train_and_test_sets(X_train, X_test)

    # -------------------------
    # 1. Window X and y
    # -------------------------
    X_train_w = Windowing.make_windows_from_X(X_train, p.window_size, p.sliding_size)
    X_test_w  = Windowing.make_windows_from_X(X_test,  p.window_size, p.sliding_size)

    y_train_w = Windowing.make_windows_from_y(y_train_s, p.window_size, p.sliding_size, task=p.task)
    y_test_w  = Windowing.make_windows_from_y(y_test_s,  p.window_size, p.sliding_size, task=p.task)

    # -------------------------
    # 2. TS2Vec encode
    # -------------------------
    ts = TS2VecEncoder(lr=train_cfg["lr"], device=device, patience=train_cfg["patience"])
    ts.fit_ts2vec(
        X_train_w.numpy(),
        hidden_dims=model_cfg["hidden_dims"],
        output_dims=model_cfg["latent_dims"],
        depth=model_cfg["depth"],
        batch_size=train_cfg["batch_size"],
        n_epochs=train_cfg["epochs"],
    )

    z_train = ts.encode(X_train_w.numpy(), pooling="mean")
    z_test  = ts.encode(X_test_w.numpy(),  pooling="mean")

    assert z_train.shape[0] == y_train_w.shape[0]
    assert z_test.shape[0]  == y_test_w.shape[0]

    # -------------------------
    # 3. Regress
    # -------------------------
    y_hat = fit_catboost_multi(z_train, y_train_w, z_test)
    rmse  = np.sqrt(mean_squared_error(y_test_w, y_hat))
    r2    = r2_score(y_test_w, y_hat)

    return rmse, r2


def run_split_ts2vec_hybrid(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: np.ndarray,
    y_test: np.ndarray,
    p: RunParams,
    model_cfg: dict,
    train_cfg: dict,
    device):
    """Option 2: split → TS2Vec separately → spherical projection → regress"""

    # -------------------------
    # 0. Split + scale
    # -------------------------
    X_lin_train, X_cyc_train = split_dataset_to_linear_and_cyclic(X_train, threshold=p.cyclic_threshold, verbose=False)
    X_lin_test  = X_test[X_lin_train.columns]
    X_cyc_test  = X_test[X_cyc_train.columns]

    y_train_s, y_test_s     = scale_train_and_test_sets(y_train, y_test)
    X_lin_train, X_lin_test = scale_train_and_test_sets(X_lin_train, X_lin_test)
    X_cyc_train, X_cyc_test = scale_train_and_test_sets(X_cyc_train, X_cyc_test)

    # -------------------------
    # 1. Window X
    # -------------------------
    X_lin_train_w = Windowing.make_windows_from_X(X_lin_train, p.window_size, p.sliding_size)
    X_cyc_train_w = Windowing.make_windows_from_X(X_cyc_train, p.window_size, p.sliding_size)
    X_lin_test_w  = Windowing.make_windows_from_X(X_lin_test,  p.window_size, p.sliding_size)
    X_cyc_test_w  = Windowing.make_windows_from_X(X_cyc_test,  p.window_size, p.sliding_size)

    # -------------------------
    # 2. Window y  ✅ CRITICAL
    # -------------------------
    y_train_w = Windowing.make_windows_from_y(y_train_s, p.window_size, p.sliding_size, task=p.task)
    y_test_w  = Windowing.make_windows_from_y(y_test_s,  p.window_size, p.sliding_size, task=p.task)

    # -------------------------
    # 3. TS2Vec encoders
    # -------------------------
    ts_lin = TS2VecEncoder(lr=train_cfg["lr"], device=device, patience=train_cfg["patience"])
    ts_lin.fit_ts2vec(X_lin_train_w.numpy(), hidden_dims=model_cfg["hidden_dims"],
                      output_dims=model_cfg["latent_dims"], depth=model_cfg["depth"],
                      batch_size=train_cfg["batch_size"], n_epochs=train_cfg["epochs"])

    ts_cyc = TS2VecEncoder(lr=train_cfg["lr"], device=device, patience=train_cfg["patience"])
    ts_cyc.fit_ts2vec(X_cyc_train_w.numpy(), hidden_dims=model_cfg["hidden_dims"],
                      output_dims=model_cfg["latent_dims"], depth=model_cfg["depth"],
                      batch_size=train_cfg["batch_size"], n_epochs=train_cfg["epochs"])

    z_lin_train = ts_lin.encode(X_lin_train_w.numpy(), pooling="mean")
    z_lin_test  = ts_lin.encode(X_lin_test_w.numpy(),  pooling="mean")
    z_cyc_train = ts_cyc.encode(X_cyc_train_w.numpy(), pooling="mean")
    z_cyc_test  = ts_cyc.encode(X_cyc_test_w.numpy(),  pooling="mean")

    # -------------------------
    # 4. Spherical projection (cyclic only)
    # -------------------------
    z_cyc_train /= np.linalg.norm(z_cyc_train, axis=1, keepdims=True)
    z_cyc_test  /= np.linalg.norm(z_cyc_test,  axis=1, keepdims=True)

    # -------------------------
    # 5. Combine + regress
    # -------------------------
    Z_train = np.concatenate([z_lin_train, z_cyc_train], axis=1)
    Z_test  = np.concatenate([z_lin_test,  z_cyc_test],  axis=1)

    assert Z_train.shape[0] == y_train_w.shape[0]
    assert Z_test.shape[0]  == y_test_w.shape[0]

    y_hat = fit_catboost_multi(Z_train, y_train_w, Z_test)
    rmse  = np.sqrt(mean_squared_error(y_test_w, y_hat))
    r2    = r2_score(y_test_w, y_hat)

    return rmse, r2

# CONFIG
model_cfg = {
    "hidden_dims": 64,
    "latent_dims": 16,
    "depth": 5,}

train_cfg = {
    "lr": 1e-3,
    "patience": 20,
    "batch_size": 16,
    "epochs": 2}, #epoch,

# RUN
rmse_plain, r2_plain = run_ts2vec_plain(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    p=p,
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    device=device,)

rmse_split, r2_split = run_split_ts2vec_hybrid(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    p=p,
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    device=device,)

print(f"TS2Vec plain        : RMSE={rmse_plain:.4f}, R2={r2_plain:.4f}")
print(f"Split + geometry    : RMSE={rmse_split:.4f}, R2={r2_split:.4f}")


In [ ]:
"dimensionality estimation"
import skdim
from sklearn.decomposition import PCA
import geomstats.geometry.hypersphere as hs
from geomstats.learning.pca import TangentPCA

# 1. Prepare Data
X_array = X.values.astype(np.float32)
X_flattened = X_array 
n_samples, ambient_dim = X_flattened.shape

# --- Intrinsic Dimension Estimation ---

# A. TwoNN & B. MLE (Geometric/Local)
# These are often the most reliable for manifold learning
d_twonn = skdim.id.TwoNN().fit_transform(X_flattened)
d_mle   = skdim.id.MLE().fit_transform(X_flattened)

# C. PCA (Global Linear) - 90% Variance threshold
pca = PCA().fit(X_flattened)
id_pca = np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.9) + 1

# D. PGA (Spherical Manifold)
# Hypersphere dim is (ambient - 1)
sphere = hs.Hypersphere(dim=ambient_dim - 1)
norms  = np.linalg.norm(X_flattened, axis=1, keepdims=True)
X_proj = X_flattened / np.where(norms == 0, 1.0, norms)

# Limit n_components to avoid memory issues in high-D
pga = TangentPCA(sphere, n_components=min(n_samples, ambient_dim, 100)) 
pga.fit(X_proj)
id_pga = np.searchsorted(np.cumsum(pga.explained_variance_ratio_), 0.9) + 1

# --- Final Decisions ---
print(f"TwoNN ID: {d_twonn:.2f}")
print(f"MLE ID:   {d_mle:.2f}")
print(f"PCA ID:   {id_pca}")
print(f"PGA ID:   {id_pga}")

# Decision Rule: 
# Usually, d_twonn or d_mle represent the 'true' manifold dimension.
# PCA/PGA represent the linear 'embedding' dimension.
final_z_dim = int(np.ceil(max(d_twonn, d_mle, id_pca, id_pga)))

print(f"\nRecommended VAE z_dim: {final_z_dim}")

In [ ]:
"kernel PCA"
from sklearn.decomposition import KernelPCA
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score

# ===== 1. Sweep embedding dimension to estimate residual variance (optional) =====
# Note: KernelPCA does not give a built-in dist_matrix_, so residual variance calculation is less straightforward.
# Here we can skip it or just look at explained variance if using linear kernel.
# For nonlinear kernels, you typically pick n_components based on domain knowledge or grid search.

# ===== 2. Fit kernel PCA with chosen embedding dimension =====
n_components = 8  # same as your Isomap example
kpca_model   = KernelPCA(n_components=n_components, kernel='rbf', gamma=0.05, fit_inverse_transform=True)
X_train_kpca = kpca_model.fit_transform(X_train)
X_test_kpca  = kpca_model.transform(X_test)

# ===== 3. Plot first 3 components (for visualization) =====
fig = plt.figure(figsize=(8,6))
ax  = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(
    X_train_kpca[:,0],
    X_train_kpca[:,1],
    X_train_kpca[:,2],
    c=y_train_np,
    cmap='tab10',
    alpha=0.8)
fig.colorbar(scatter, label='Class/Label')
ax.set_title("Kernel PCA 3D Embedding")
plt.show()

# ===== 4. RandomForestClassifier on KPCA embedding =====
rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train_kpca, y_train_np)
y_pred_class = rf_classifier.predict(X_test_kpca)
accuracy     = accuracy_score(y_test_np, y_pred_class)
rmse_class   = np.sqrt(mean_squared_error(y_test_np, y_pred_class))
print(f"Kernel PCA accuracy (classif.): {accuracy:.3f}")
print(f"Kernel PCA RMSE (classif.): {rmse_class:.3f}")

# ===== 5. RandomForestRegressor on KPCA embedding =====
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train_kpca, y_train_np)
y_pred_reg = rf_regressor.predict(X_test_kpca)
r2        = r2_score(y_test_np, y_pred_reg)
rmse_reg  = np.sqrt(mean_squared_error(y_test_np, y_pred_reg))
print(f"Kernel PCA R² (regression): {r2:.3f}")
print(f"Kernel PCA RMSE (regression): {rmse_reg:.3f}")


In [ ]:
"isomap"
from sklearn.manifold import Isomap
import warnings
from scipy.sparse import SparseEfficiencyWarning
warnings.simplefilter('ignore', SparseEfficiencyWarning)

# ===== 1. plot residual variance to estimate intrinsic dimensionality (finds the best dim)
res_vars       = []
embedding_dims = range(1, 11)  # try 1D to 10D embeddings
for dim in embedding_dims:
    iso = Isomap(n_neighbors=20, n_components=dim)
    iso.fit(X_train)
    # residual variance = 1 - R^2 between graph distances and embedding distances
    dist_graph = iso.dist_matrix_
    dist_emb   = np.linalg.norm(iso.embedding_[:, None, :] - iso.embedding_[None, :, :], axis=2)
    r2         = np.corrcoef(dist_graph.ravel(), dist_emb.ravel())[0,1]**2
    res_vars.append(1 - r2)
plt.plot(embedding_dims, res_vars, marker='o')
plt.xlabel("Embedding dimension")
plt.ylabel("Residual variance")
plt.title("Estimate intrinsic dimensionality")
plt.show()

# ===== 2. plot isomap of training data (uses the best estimated dim from above)
isomap_model   = Isomap(n_neighbors=20, n_components=5)
X_train_isomap = isomap_model.fit_transform(X_train)
X_test_isomap  = isomap_model.transform(X_test)

fig     = plt.figure(figsize=(8,6))
ax      = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(
    X_train_isomap[:,0],
    X_train_isomap[:,1],
    X_train_isomap[:,2],
    c=y_train_np,
    cmap='tab10',
    alpha=0.8)
fig.colorbar(scatter, label='Class/Label')  # attach to figure
ax.set_title("Isomap 3D Embedding")
plt.show()


In [ ]:
"isomap results"
rf_classifier= RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train_isomap, y_train_np)          # train on Isomap embedding
y_pred_class = rf_classifier.predict(X_test_isomap)          # predict test labels
accuracy     = accuracy_score(y_test_np, y_pred_class)
print(f"Isomap accuracy (classif.): {accuracy:.3f}")
rmse         = np.sqrt(mean_squared_error(y_test_np, y_pred_class))
print(f"Isomap RMSE (classif.): {rmse:.3f}")

# ====
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train_isomap, y_train_np)           # train on Isomap embedding
y_pred_reg   = rf_regressor.predict(X_test_isomap)      # predict on test embedding
r2           = r2_score(y_test_np, y_pred_reg)
print(f"Isomap R² (regression): {r2:.3f}")
test_rmse    = np.sqrt(mean_squared_error(y_test_np, y_pred_reg))
print(f"Isomap RMSE (regression): {test_rmse:.3f}")


In [ ]:
"new metrics"
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, accuracy_score, r2_score

def propagate_knn(Z_latent, y_true, missing_frac=0.2, k=15, tau=1.0):
    """Perform kNN label propagation on latent vectors."""
    num_rows = Z_latent.shape[0]
    dist_matrix = np.zeros((num_rows, num_rows))
    
    # compute L2 distances
    for i in range(num_rows):
        u = Z_latent[i]
        for j in range(i, num_rows):
            v = Z_latent[j]
            d = np.linalg.norm(u - v)
            dist_matrix[i, j] = d
            dist_matrix[j, i] = d

    # mask random fraction
    num_missing = int(missing_frac * num_rows)
    missing_idx = np.random.choice(num_rows, size=num_missing, replace=False)
    y_input     = y_true.copy().astype(float)
    y_input[missing_idx] = np.nan

    # kNN neighbors
    neighbors_idx = np.argsort(dist_matrix, axis=1)[:, 1:k+1]

    # edge weights
    def compute_weights(dists):
        w = np.exp(-dists / tau)
        return w / w.sum()

    # propagate labels
    y_prop = y_input.copy()
    for i in range(num_rows):
        if np.isnan(y_input[i]):
            neigh     = neighbors_idx[i]
            neigh_y   = y_input[neigh]
            mask      = ~np.isnan(neigh_y)
            if np.sum(mask) == 0:
                continue
            neigh_y   = neigh_y[mask]
            neigh_d   = dist_matrix[i, neigh][mask]
            w         = compute_weights(neigh_d)
            y_prop[i] = np.sum(w * neigh_y)

    return y_prop, missing_idx

def evaluate_metrics(y_true, y_pred, missing_idx):
    """Compute RMSE, MAE, Accuracy, Spearman correlation, and R² for masked points."""
    y_true_masked = y_true[missing_idx]
    y_pred_masked = y_pred[missing_idx]
    mask_valid    = ~np.isnan(y_pred_masked)
    
    if mask_valid.sum() == 0:
        return {k: np.nan for k in ["rmse","mae","acc","spearman","r2"]}
    
    y_true_valid = y_true_masked[mask_valid]
    y_pred_valid = y_pred_masked[mask_valid]


    def safe_spearmanr(x, y):
        if np.std(x) == 0 or np.std(y) == 0:
            return np.nan
        return spearmanr(x, y).correlation

    metrics = {
        "rmse": np.sqrt(mean_squared_error(y_true_valid, y_pred_valid)),
        "mae": mean_absolute_error(y_true_valid, y_pred_valid),
        "acc": accuracy_score(y_true_valid, np.round(y_pred_valid).astype(int)),
        "spearman": safe_spearmanr(y_true_valid, y_pred_valid),
        "r2": r2_score(y_true_valid, y_pred_valid)
    }
    return metrics

def format_metrics(metrics: dict) -> dict:
    """Format all float metrics to 3 decimal places, keep nan as-is."""
    return {k: (f"{v:.3f}" if isinstance(v, float) and not np.isnan(v) else v)
            for k, v in metrics.items()}

# --- Hybrid AE ---
y_prop_hybrid, missing_idx = propagate_knn(z_hybrid_encoded_np, df.iloc[:,1].values)
metrics_hybrid = evaluate_metrics(df.iloc[:,1].values, y_prop_hybrid, missing_idx)
print("Hybrid metrics:", format_metrics(metrics_hybrid))

# --- Full Euclidean AE ---
y_prop_euc, missing_idx = propagate_knn(z_euclid_full_np, df.iloc[:,1].values)
metrics_euc = evaluate_metrics(df.iloc[:,1].values, y_prop_euc, missing_idx)
print("Full Euclidean metrics:", format_metrics(metrics_euc))

# --- Baseline (mean predictor) ---
y_true      = df.iloc[:,1].values
num_rows    = len(y_true)
num_missing = int(0.2 * num_rows)
np.random.seed(42)
missing_idx = np.random.choice(num_rows, size=num_missing, replace=False)

y_baseline = y_true.copy().astype(float)
y_baseline[missing_idx] = np.nan
mean_label = np.nanmean(y_baseline)
y_baseline[np.isnan(y_baseline)] = mean_label

metrics_baseline = evaluate_metrics(y_true, y_baseline, missing_idx)
print("Baseline metrics:", format_metrics(metrics_baseline))
